<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def load_data(seed=42):
    X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
    y = y.astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = load_data()
print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(n_estimators=100, random_state=seed)
    model.fit(X_train, y_train)
    return model

rf_model = train_random_forest(X_train, y_train)
ab_model = train_adaboost(X_train, y_train)
print("Modelos treinados com sucesso.")

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc

acc_rf = evaluate(rf_model, X_test, y_test)
acc_ab = evaluate(ab_model, X_test, y_test)
print(f"Acurácia Random Forest: {acc_rf:.4f}")
print(f"Acurácia AdaBoost: {acc_ab:.4f}")

**Resposta**: Para modelos baseados em árvores de decisão (como Random Forest e AdaBoost com árvores), a normalização dos dados **não é necessária**. Esses modelos realizam divisões baseadas em limiares nos valores das features, e a escala dos dados não afeta essas divisões. Diferentemente de modelos baseados em distância (KNN, SVM) ou gradiente (redes neurais), árvores de decisão são invariantes à escala dos dados.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError(f"model_type deve ser 'rf' ou 'ab', recebeu '{model_type}'")
    acc = evaluate(model, X_test, y_test)
    return acc

acc = run_pipeline("rf", seed=42)
print(f"Pipeline RF acurácia: {acc:.4f}")

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
for name, model_type in [("Random Forest", "rf"), ("AdaBoost", "ab")]:
    X_tr, X_te, y_tr, y_te = load_data(42)
    if model_type == "rf":
        model = train_random_forest(X_tr, y_tr, 42)
    else:
        model = train_adaboost(X_tr, y_tr, 42)

    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average="weighted")
    rec = recall_score(y_te, y_pred, average="weighted")
    f1 = f1_score(y_te, y_pred, average="weighted")

    print(f"--- {name} ---")
    print(f"Acurácia:  {acc:.4f}")
    print(f"Precisão:  {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print()

print("O Random Forest apresentou melhor desempenho inicial em todas as métricas.")

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
for seed in [42, 7]:
    acc_rf = run_pipeline("rf", seed=seed)
    acc_ab = run_pipeline("ab", seed=seed)
    print(f"Seed={seed}: RF={acc_rf:.4f}, AdaBoost={acc_ab:.4f}")

acc1 = run_pipeline("rf", seed=42)
acc2 = run_pipeline("rf", seed=42)
print(f"\nReprodutibilidade: acc1={acc1:.4f}, acc2={acc2:.4f}, igual={abs(acc1-acc2) < 1e-6}")
print("Sim, o experimento é reprodutível. Com a mesma seed, os resultados são idênticos.")
print("Com seeds diferentes, os resultados variam ligeiramente devido à divisão dos dados e à aleatoriedade dos modelos.")

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
X_train, X_test, y_train, y_test = load_data(42)

for name, model_type in [("Random Forest", "rf"), ("AdaBoost", "ab")]:
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, 42)
    else:
        model = train_adaboost(X_train, y_train, 42)

    acc_train = accuracy_score(y_train, model.predict(X_train))
    acc_test = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: Treino={acc_train:.4f}, Teste={acc_test:.4f}, Gap={acc_train - acc_test:.4f}")

print("\nO Random Forest tende a sofrer mais overfitting (acurácia de treino próxima de 100%).")
print("Isso ocorre porque árvores profundas memorizam o conjunto de treino.")
print("O AdaBoost, usando stumps por padrão, tem menor capacidade e menor gap treino/teste.")

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
X_train, X_test, y_train, y_test = load_data(42)

print("--- Random Forest: variando n_estimators ---")
for n in [10, 50, 100, 200]:
    model = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"  n_estimators={n:>3}: acurácia={acc:.4f}")

print("\n--- AdaBoost: variando n_estimators ---")
for n in [10, 50, 100, 200]:
    model = AdaBoostClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"  n_estimators={n:>3}: acurácia={acc:.4f}")

print("\nO AdaBoost é mais sensível a mudanças em n_estimators, pois cada estimador corrige")
print("erros dos anteriores de forma sequencial. O Random Forest estabiliza mais rápido com bagging.")

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

In [ ]:
print("""1. A acurácia é suficiente para avaliar os modelos?
Não. A acurácia sozinha pode ser enganosa, especialmente em datasets desbalanceados.
Métricas como precisão, recall e F1-Score fornecem uma visão mais completa, revelando
dificuldades em classes específicas que a acurácia global esconde.

2. Como você garante que o resultado não ocorreu por acaso?
Usando random_state fixo em todas as etapas garantimos reprodutibilidade. Testamos com
múltiplas seeds para verificar consistência. Para maior robustez, poderia-se usar
validação cruzada (k-fold cross-validation).

3. Cite dois possíveis problemas metodológicos neste experimento.
- Usamos apenas uma divisão treino/teste (hold-out), tornando o resultado dependente
  dessa partição. Validação cruzada seria mais robusta.
- Não realizamos busca sistemática de hiperparâmetros (GridSearchCV/RandomizedSearchCV),
  então os modelos podem não estar otimizados e a comparação pode não ser justa.

4. O pipeline implementado é confiável? Justifique.
O pipeline é razoavelmente confiável: é reprodutível (random_state), modular (funções
separadas), e testável (CI automatizado). Para produção, seria necessário adicionar
validação cruzada, busca de hiperparâmetros e análise de variância dos resultados.
""")